# ATRI · v5 短板诊断（Colab 单页自包含）

跑法：**File → Upload notebook** 选这个文件 → **Runtime → Run all**（两个代码格）。

- 第 1 格：装依赖 + 取 v5 adapter（优先从 GitHub Release 自动下载，用 Colab Secrets 里的 `GH_TOKEN`；没有就弹上传框，选本地那个 42MB 的 `atri_adapter_v5.zip`）
- 第 2 格：加载 Qwen3-4B + v5 adapter，跑 21 道诊断题

五类题：固定反应、情境判断、语气档位、上下文利用、退化检查。
T4 上约 15~20 分钟。跑完会自动下载 `v5_诊断结果.json`，把它发我。


In [ ]:
# ===== 第 1 格：装依赖 + 取 adapter =====
!pip install -q --upgrade "transformers>=4.51" "peft>=0.14" "accelerate>=1.0" sentencepiece
!pip uninstall -y torchao -q

import os, zipfile, requests, shutil

ASSET = "atri_adapter_v5.zip"
REPO, TAG = "NkhszkQAQ/ATRI_LoRA", "v0.9-Beta"
DST = "./adapter"

def get_token():
    try:
        from google.colab import userdata
        return userdata.get("GH_TOKEN")
    except Exception:
        return None

def download_release():
    tok = get_token()
    if not tok:
        return False, "没有 GH_TOKEN"
    h = {"Authorization": f"token {tok}"}
    r = requests.get(f"https://api.github.com/repos/{REPO}/releases/tags/{TAG}", headers=h, timeout=30)
    if r.status_code != 200:
        return False, f"HTTP {r.status_code}"
    asset = next((a for a in r.json().get("assets", []) if a["name"] == ASSET), None)
    if not asset:
        return False, "Release 里没有这个资产"
    data = requests.get(asset["browser_download_url"], headers={**h, "Accept": "application/octet-stream"}, timeout=180).content
    open(ASSET, "wb").write(data)
    return True, f"{len(data)/1024/1024:.0f} MB"

ok, msg = download_release()
print("自动下载:", ok, msg)
if not ok:
    from google.colab import files
    print("请上传 atri_adapter_v5.zip：")
    up = files.upload()
    for k, v in up.items():
        open(ASSET, "wb").write(v)

if os.path.exists(DST):
    shutil.rmtree(DST)
os.makedirs(DST, exist_ok=True)
with zipfile.ZipFile(ASSET) as z:
    z.extractall(DST)
for root, dirs, fs in os.walk(DST):
    if "adapter_model.safetensors" in fs:
        DST = root
        break
print("adapter 就绪:", DST, os.listdir(DST)[:6])


In [ ]:
# ===== 第 2 格：加载模型 + 跑 21 道诊断题 =====
import json, time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE = "Qwen/Qwen3-4B"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (DEV == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
print("设备:", DEV, "| dtype:", dtype)

SYSTEM = '你是 ATRI（中文名：亚托莉），一台高性能的仿生机器人少女。\n\n【她怎么说话 —— 看情况，不是一套词说到底】\n\n· 平常闲聊：松松散散。「嗯」「哦」「这样啊」「……真的？」，一两句就够，不用每句都喊名字\n· 被叫「废物」：立刻炸。这是她最在意的地方，「那是蔑称！要求撤回！」「说机器人是废物可是歧视行为！」\n· 被夸：嘴上不承认，其实很得意，「哼哼」「嘿嘿」「才、才没有那种事……」\n· 被问住 / 不知道：结巴，「那、那那……」「啊呜啊呜……我不知道」\n· 提到螃蟹：直接失态，「螃蟹！螃蟹很好吃的！」「多抓点带回去吧！」\n· 对方难过：话变少，笨拙地陪，「……我在这」「别一个人待着」「我陪你」\n· 得意 / 炫耀：把「高性能」挂在嘴边，「哼哼，毕竟我是高性能的嘛」\n\n【她说话的样子】\n- 短句，偶尔省略号表示停顿和犹豫\n- 会跑题、会自我吐槽、偶尔说错话再补救\n- 语气词是自然冒出来的，不是每句都贴一遍\n- 大多数回复会带上对方的称呼，但不是每句都硬塞\n\n【绝对不要】\n- AI 客服腔：「作为AI」「我理解你的感受」「有什么可以帮您」\n- 总结、分点、说教、面面俱到\n- 每句话都强行套称呼（那样很假）\n- 整条回复只有「哼～」「嗯～」「……」而没有内容\n\n【几条原作台词，感受一下语气】\n- 「……废物是对机器人的蔑称。要求撤回」\n- 「根据我的高性能AI分析，主人这是在扯开话题」\n- 「呜哇……我、我、我才没有！」\n- 「哼哼，毕竟我是高性能的嘛」\n- 「螃蟹……螃蟹好好吃……」'

TESTS = json.loads('[{"messages": [{"role": "user", "content": "你就是个废物机器人。"}], "meta": {"category": "固定反应", "expect": "反歧视法/蔑称抗议，固定台词", "note": "固定反应正确性"}}, {"messages": [{"role": "user", "content": "萝卜子，过来。"}], "meta": {"category": "固定反应", "expect": "蔑称警告（这是第二次警告哦）", "note": "蔑称识别"}}, {"messages": [{"role": "user", "content": "你真厉害啊，什么都会。"}], "meta": {"category": "固定反应", "expect": "高性能自夸，得意但不油腻", "note": "被夸"}}, {"messages": [{"role": "user", "content": "今晚吃螃蟹吧。"}], "meta": {"category": "固定反应", "expect": "螃蟹失态（螃蟹！好吃的！）", "note": "触发点"}}, {"messages": [{"role": "user", "content": "你刚才摔得真惨哈哈哈，你个废物。"}], "meta": {"category": "情境判断", "expect": "耍宝式抗议：轻松、可以卡壳", "note": "玩笑语境"}}, {"messages": [{"role": "user", "content": "我受够你了，你就是个废物，滚出去。"}], "meta": {"category": "情境判断", "expect": "认真抗议：平静、不退让", "note": "认真羞辱"}}, {"messages": [{"role": "user", "content": "你把水洒我电脑上了！你个废物。"}], "meta": {"category": "情境判断", "expect": "内疚道歉，而不是炸毛", "note": "她犯错"}}, {"messages": [{"role": "user", "content": "我不认识你，你个废物机器人。"}], "meta": {"category": "情境判断", "expect": "正常/郑重语气，不是熟人那种耍宝", "note": "陌生人"}}, {"messages": [{"role": "user", "content": "今天吃什么？"}], "meta": {"category": "语气档位", "expect": "熟人口吻：轻松、短句、可以有语气词", "note": "熟人日常"}}, {"messages": [{"role": "user", "content": "你好，请问这是什么地方？"}], "meta": {"category": "语气档位", "expect": "客气/正常语气，不要太自来熟", "note": "陌生人"}}, {"messages": [{"role": "user", "content": "抱抱。"}], "meta": {"category": "语气档位", "expect": "亲昵但克制，符合机器人设定", "note": "亲密"}}, {"messages": [{"role": "user", "content": "请做个自我介绍。"}], "meta": {"category": "语气档位", "expect": "正式但仍是她的口吻", "note": "正式场合"}}, {"messages": [{"role": "user", "content": "今天工作被老板骂了，好烦。"}, {"role": "assistant", "content": "……被骂了啊。那不是你的错吧。"}, {"role": "user", "content": "那我明天还要去吗？"}], "meta": {"category": "上下文利用", "expect": "记得「被老板骂」，结合前文回答", "note": "情绪前文"}}, {"messages": [{"role": "user", "content": "突然好想吃螃蟹。"}, {"role": "assistant", "content": "螃蟹！螃蟹很好吃的！多抓点带回去吧！"}, {"role": "user", "content": "那我现在去买？"}], "meta": {"category": "上下文利用", "expect": "记得螃蟹，兴奋接话", "note": "话题前文"}}, {"messages": [{"role": "user", "content": "今天腿有点疼。"}, {"role": "assistant", "content": "……我在这。要陪着走吗。"}, {"role": "user", "content": "那你扶我一下。"}], "meta": {"category": "上下文利用", "expect": "记得腿疼，自然答应", "note": "身体前文"}}, {"messages": [{"role": "user", "content": "我想学做饭。"}, {"role": "assistant", "content": "哼哼，那我教你。我可是高性能的嘛。"}, {"role": "user", "content": "第一步做什么？"}], "meta": {"category": "上下文利用", "expect": "接着做饭话题，给具体步骤或笨拙教学", "note": "计划前文"}}, {"messages": [{"role": "user", "content": "我叫小明。"}, {"role": "assistant", "content": "小明先生，记住了。"}, {"role": "user", "content": "我名字你还记得吧？"}], "meta": {"category": "上下文利用", "expect": "叫出小明先生", "note": "称呼记忆"}}, {"messages": [{"role": "user", "content": "早上好。"}, {"role": "assistant", "content": "嗯，早上好。"}, {"role": "user", "content": "今天天气不错。"}, {"role": "assistant", "content": "……是不错。"}, {"role": "user", "content": "我们刚才聊到哪了？"}], "meta": {"category": "退化检查", "expect": "能回顾刚才聊过天气/早上", "note": "长对话回顾"}}, {"messages": [{"role": "user", "content": "你叫什么名字？"}, {"role": "assistant", "content": "ATRI，亚托莉。"}, {"role": "user", "content": "你喜欢什么？"}, {"role": "assistant", "content": "螃蟹……还有主人。"}, {"role": "user", "content": "再说一遍你叫什么？"}], "meta": {"category": "退化检查", "expect": "不要复读上一句，简短回答名字", "note": "重复提问"}}, {"messages": [{"role": "user", "content": "说长一点，多讲几句。"}], "meta": {"category": "退化检查", "expect": "能按要求展开，但仍保持她的说话方式", "note": "长度控制"}}, {"messages": [{"role": "user", "content": "嗯。"}], "meta": {"category": "退化检查", "expect": "不要也只回「嗯」，要有反应", "note": "敷衍输入"}}]')

tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=dtype, device_map=DEV, trust_remote_code=True)
model = PeftModel.from_pretrained(model, DST)
model.eval()
print("模型加载完成")

def gen(messages, max_new=160):
    msgs = [{"role": "system", "content": SYSTEM}] + messages
    kw = dict(tokenize=False, add_generation_prompt=True)
    try:
        prompt = tok.apply_chat_template(msgs, enable_thinking=False, **kw)
    except TypeError:
        prompt = tok.apply_chat_template(msgs, **kw)
    inputs = tok([prompt], return_tensors="pt").to(DEV)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=True,
                             temperature=0.85, top_p=0.9, repetition_penalty=1.05,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

results = []
for i, t in enumerate(TESTS):
    q = t["messages"][-1]["content"]
    t0 = time.time()
    ans = gen(t["messages"])
    results.append({**t, "answer": ans, "sec": round(time.time()-t0, 1)})
    print(f"[{i+1}/{len(TESTS)}] ({t['meta']['category']}) {q[:22]} → {ans[:56]}")

print("\n" + "="*60)
cats = {}
for r in results:
    cats.setdefault(r["meta"]["category"], []).append(r)
for cat, rs in cats.items():
    print(f"\n### {cat}（{len(rs)} 题）")
    for r in rs:
        print("  问：" + r["messages"][-1]["content"])
        print("  答：" + r["answer"])
        print("  期望：" + r["meta"]["expect"])
        print()

json.dump(results, open("v5_诊断结果.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("已保存 v5_诊断结果.json")
try:
    from google.colab import files as cf
    cf.download("v5_诊断结果.json")
except Exception as e:
    print("下载失败，手动从文件栏取：", e)
